## Export Code Without Outputs

In [ ]:
!jupyter nbconvert --to notebook --ClearOutputPreprocessor.enabled=True --output=Pedestrian_model_clean.ipynb Pedestrian_model.ipynb

## Import and Setup

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import explained_variance_score, r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold, GroupKFold
import xgboost as xgb
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import shap
from PyALE import ale
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patheffects as pe
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
import contextily as ctx
import geopandas as gpd


### Setup

In [ ]:
mpl.rcParams.update({
    "font.family": "serif",  # use 'serif' or 'Times New Roman'
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 300,
    "savefig.dpi": 300, 
    "axes.linewidth": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "lines.linewidth": 1.5,
})

## Data Preparation and Visualization

### Read and Process Data

In [ ]:
file_path = 'Input_Data/final_pedestrian_data.csv'
df = pd.read_csv(file_path, parse_dates=['DateHour'])

X = df.drop(columns=['DeviceID', 'DateHour', 'Latitude', 'Longitude', 'geometry', 'HourlyPedCount',
                     'Major_Push_DontWalk', 'Major_Push_Walk', 'Minor_Push_DontWalk', 'Minor_Push_Walk',
                     'Pct_cycleways_in_radius', 'Major_Max_speed_mph', 'Minor_Max_speed_mph',
                    'pedestrian_incidents', 'pedalcycle_incidents', 'fatal_crashes',
                    'Violent Crimes', 'Property Crimes', 'Drug & Alcohol Related', 'Public Order & Nuisance Offenses',
                    'Traffic-Related Incidents', 'Juvenile & Family Offenses', 'Mental Health / Welfare / Assistance', 
                    'Administrative / Other / Unclassified', 'Diversion / Prevention', 'total_incidents',
                     ])
X = X.apply(pd.to_numeric, errors='coerce').astype(float)
Y = df['HourlyPedCount'].astype(float)

In [ ]:
rename_dict = {
    "Minor_PedWalk_Count": "Walk Intervals (Minor Approach)",
    "Minor_Push_Total": "Push Button Events (Minor Approach)",
    "Total_Push": "Total Push Button Events",
    "Major_Push_Total": "Push Button Events (Major Approach)",
    "Major_PedWalk_Count": "Walk Intervals (Major Approach)",
    "Education": "Education POIs",
    "Average_Waiting_Time": "Average Pedestrian Wait Time (s)",
    "Emp_Education": "Edu Sector Employees",
    "TotalEmployees_WithinRadius": "Total Employees",
    "D5BE": "Working Age Population Within 45-Minute Transit Commute",
    "Maximum_Waiting_Time": "Max Pedestrian Wait Time (s)",
    "temp": "Temperature (°C)",
    "D5BR": "Jobs Within 45-Minute Transit Commute",
    "D1D": "Activity Density",
    "Leisure": "Leisure POIs",
    "AvgCycleLen_s": "Average Cycle Length (s)",
    "AvgSpeed_y": "Average Vehicle Speed (km/h)",
    "Parking": "Parking Facilities",
    "D5AE": "Working Age Population within 45-Minute Auto Commute",
    "Pct_drive_alone": "Percentage of Drive Alone Commuters",
    "Emp_Government, Social, & Advocacy Svcs": "Gov/Social Jobs",
    "Ac_Unpr": "Block Size (Ac)",
    "Consumer Svcs": "Consumer Services",
    "Shops": "Shop POIs",
    "D5CE": "Auto Access Ratio (WAP)",
    "D4D": "Transit Service Density (service/sq. km)",
    "D3B": "Street Intersection Density",
    "D2C_WREMLX": "Household Workers per Job Equilibrium Index",
    "Pct_above_2x_poverty": "Percenatage of Population Above 2x Poverty Line",
    "Pct_below_poverty": "Percenatage of Population Below Poverty Line",
    "Pop_within_402m": "Population within Quarter Mile",
    "NatWalkInd": "National Walking Index",
    "Pct_long_commute": "Percentage of Long Commuters"
}

X = X.rename(columns=rename_dict)
X['Average Vehicle Speed (km/h)'] = X['Average Vehicle Speed (km/h)'] * 1.60934

X['has_transit_access'] = (X['Transit Service Density (service/sq. km)'] != 9999).astype(int)
X['Transit Service Density (service/sq. km)'] = X['Transit Service Density (service/sq. km)'].replace(9999, 0)
X['Transit Service Density (service/sq. km)'] = X['Transit Service Density (service/sq. km)'] / 2.58999

X['Percentage of Long Commuters'] = X['Percentage of Long Commuters'] * 100
X['Percentage of Drive Alone Commuters'] = X['Percentage of Drive Alone Commuters'] * 100
X['Pct_sidewalks_in_radius'] = X['Pct_sidewalks_in_radius'] * 100

X = X.replace([np.inf, -np.inf], np.nan).fillna(X.mean())

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

### Parity Plots

In [ ]:
combined_X = pd.concat([X_train, X_test])
combined_Y = pd.concat([Y_train, Y_test])

green_idx = combined_Y.index 

xvals = combined_Y.loc[green_idx]
yvals = combined_X.loc[green_idx, 'Total Push Button Events']

refline = np.linspace(0, 1_000, 1_000)

fig, (ax_full, ax_zoom) = plt.subplots(
    1, 2, figsize=(10, 4), sharex=False, sharey=False
)

ax_full.scatter(xvals, yvals, color='green', alpha=0.5,
                label='Retained Samples (Non-Outliers)')
ax_full.plot(refline, refline, color='blue', label='45° line')

ax_full.set_xlabel('Actual Pedestrian Volume Values')
ax_full.set_ylabel('Total Push Button Events')
ax_full.set_xlim(0, 1_000)
ax_full.set_ylim(0, 1_000)
ax_full.set_title('Full Range (0–1,000)')

ax_zoom.scatter(xvals, yvals, color='green', alpha=0.5)
ax_zoom.plot(refline, refline, color='blue')

ax_zoom.set_xlim(0, 100)
ax_zoom.set_ylim(0, 100)
ax_zoom.set_xlabel('Actual Pedestrian Volume Values')
ax_zoom.set_ylabel('Total Push Button Events')
ax_zoom.set_title('Zoomed-in (0–100)')

plt.tight_layout(w_pad=2)
plt.show()

In [ ]:
LAT, LON = "Latitude", "Longitude"

df = pd.read_csv("Input_Data/final_pedestrian_data.csv")

agg = (df.groupby("DeviceID")
         .agg(mean_push_count          = ("Total_Push",      "mean"),
              mean_pedcount      = ("HourlyPedCount",  "mean"),
              **{LAT: (LAT, "first"), LON: (LON, "first")})
         .reset_index())

gdf = (gpd.GeoDataFrame(
          agg,
          geometry=gpd.points_from_xy(agg[LON], agg[LAT]),
          crs="EPSG:4326")
       .to_crs(3857))


bins   = [0,10,20,30,40,50,75,100,150,np.inf]
labels = ["0‒10","10‒20","20‒30","30‒40","40‒50",
          "50‒75","75‒100","100‒150","150+"]

cmap = LinearSegmentedColormap.from_list(
    "gray_to_green_strong",
    ["#d9d9d9", "#a1d99b", "#31a354", "#006d2c", "#00441b"],
    N=len(labels)
)
norm = mpl.colors.BoundaryNorm(bins, cmap.N)

def add_legend(ax, title):
    handles = [Line2D([0], [0], marker='o', linestyle='',
                      markersize=6,
                      markerfacecolor=cmap(norm((bins[i]+bins[i+1])/2)),
                      markeredgecolor='k',
                      label=lab)
               for i, lab in enumerate(labels)]
    ax.legend(handles=handles, title=title,
              loc="upper right", frameon=False,
              fontsize=12, title_fontsize=13)

fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(14, 11), dpi=300)

for ax, metric, title in [
    (ax1, "mean_pedcount", "Mean Hourly Pedestrian Volume"),
    (ax2, "mean_push_count",     "Mean Push Button Events")
]:
    sc = ax.scatter(
        gdf.geometry.x, gdf.geometry.y,
        s=85,
        c=gdf[metric],
        cmap=cmap, norm=norm,
        edgecolor="black", linewidth=0.35, alpha=0.95,
        zorder=3
    )

    sc.set_path_effects([
        pe.Stroke(linewidth=2.2, foreground="white"),
        pe.Normal()
    ])

    try:
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Voyager, alpha=0.9)
    except AttributeError:
        ctx.add_basemap(ax, source=ctx.providers.Esri.WorldGrayCanvas, alpha=0.9)

    add_legend(ax, title)
    ax.set_axis_off()

plt.tight_layout(w_pad=2)
plt.show()

## Model

### Optimize XGBoost Model

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300, 400, 500, 600, 700],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 9, 11],
    'min_child_weight': [1, 3, 5, 7],
    'subsample': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.001, 0.01, 0.1, 1, 10, 20, 50],
    'reg_lambda': [0.001, 0.005, 0.01, 0.1, 1, 10, 100, 500],
    'gamma': [0, 0.01, 0.1, 0.2, 0.5, 1]
}

xgb_model = xgb.XGBRegressor(objective='count:poisson', random_state=42)

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=50,
    scoring='r2',
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, Y_train)

print("Best Parameters:", random_search.best_params_)
print("Best R-squared Score from CV:", random_search.best_score_)

In [ ]:
best_xgb_model = random_search.best_estimator_

Y_pred_model = best_xgb_model.predict(X_test)

Y_pred_model = np.round(Y_pred_model)

Y_test_filtered = Y_test[(Y_test > 1)]
Y_pred_model_filtered = Y_pred_model[(Y_test > 1)]

rmse = root_mean_squared_error(Y_test, Y_pred_model)
r2 = r2_score(Y_test, Y_pred_model)
mae = mean_absolute_error(Y_test, Y_pred_model)
print("Root Mean Squared Error on Test Set:", rmse)
print("R-squared on Test Set:", r2)
print("Mean Abslote Error on Test Set:", mae)
explained_variance = explained_variance_score(Y_test, Y_pred_model)

print("Explained Variance Score:", explained_variance)

### Best XGBoost Model

In [ ]:
def run_best_model(x_train, y_train):
    model = xgb.XGBRegressor(
        subsample=1.0,
        n_estimators=400,
        min_child_weight=3,
        max_depth=3,
        learning_rate=0.2,
        colsample_bytree=0.6,
        reg_alpha=0.001,
        reg_lambda=10,
        gamma=0,
        objective="count:poisson",
        random_state=42
    )
    model.fit(x_train, y_train)

    return model

### Find Best Model

In [ ]:
numeric_features  = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns

# Preprocessor: scale numerics, one-hot encode categoricals
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="passthrough",
)

models = {
    "LinearRegression": LinearRegression(),
    "RBF-SVM":          SVR(kernel="rbf", C=10, epsilon=0.1),
    "RandomForest":     RandomForestRegressor(n_estimators=300, max_depth=None, n_jobs=-1, random_state=42),
    "MLP (2×64)":       MLPRegressor(hidden_layer_sizes=(64, 64), learning_rate_init=0.003,
                                     max_iter=500, random_state=42),
    "XGBoost":          xgb.XGBRegressor(
        subsample=1.0,
        n_estimators=400,
        min_child_weight=3,
        max_depth=3,
        learning_rate=0.2,
        colsample_bytree=0.6,
        reg_alpha=0.001,
        reg_lambda=10,
        gamma=0,
        objective="count:poisson",
        random_state=42
    ),
}

for name, model in models.items():
    pipe = Pipeline(steps=[("prep", preprocessor), ("model", model)])
    pipe.fit(X_train, Y_train)
    y_pred = pipe.predict(X_test)
    mae  = mean_absolute_error(Y_test, y_pred)
    r2   = r2_score(Y_test, y_pred)
    print(f"{name:15s} | MAE: {mae:8.5f} | R²: {r2:6.5f}")

### K-Fold Cross Validation

In [ ]:
def kfold_cv_xgb(X, y, n_splits=5, shuffle=True, random_state=42):
    kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)

    r2_scores, mae_scores = [], []
    models = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X), start=1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        model = run_best_model(X_tr, y_tr)
        y_pred = model.predict(X_va)

        r2  = r2_score(y_va, y_pred)
        mae = mean_absolute_error(y_va, y_pred)

        r2_scores.append(r2)
        mae_scores.append(mae)
        models.append(model)

        print(f"Fold {fold}: R²={r2:.4f} | MAE={mae:.4f}")

    print(f"\nR²:   mean={np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")
    print(f"MAE: mean={np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")

    final_model = run_best_model(X, y)

    return {
        "models_per_fold": models,
        "r2_per_fold": r2_scores,
        "mae_per_fold": mae_scores,
        "final_model": final_model
    }

results = kfold_cv_xgb(X.values if hasattr(X, "values") else X, Y.values if hasattr(Y, "values") else Y, n_splits=5)

## Analysis

### SHAP and ALE Analysis

#### SHAP Values

In [ ]:
model = run_best_model(X,Y)

explainer = shap.TreeExplainer(model)
shap_values = explainer(X)

mean_shap_values = np.abs(shap_values.values).mean(axis=0)

feature_importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Mean SHAP Value": mean_shap_values
}).sort_values(by="Mean SHAP Value", ascending=False)

print("SHAP Feature Importance:")
print(feature_importance_df[:20])

#### SHAP and ALE Plots

In [ ]:
best_xgb_model = run_best_model(X_train, Y_train)

explainer = shap.TreeExplainer(best_xgb_model)
shap_values = explainer(X)

abs_mean  = np.abs(shap_values.values).mean(axis=0)
top_idx   = np.argsort(abs_mean)[-20:][::-1]
top_feats = X.columns[top_idx]

for feat, idx in zip(top_feats, top_idx):
    x = X.iloc[:, idx]
    ys = shap_values.values[:, idx]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3))

    print(f"SHAP and ALE Interpretation for {feat}")

    ax1.scatter(x, ys, s=10, alpha=0.4)
    ax1.axhline(y=0, color="red", linestyle="--", linewidth=1, zorder=0)

    if len(feat) > 35:
        font_size = 7
    elif len(feat) > 30:
        font_size = 8
    else:
        font_size = 9

    ax1.set_xlabel(feat, fontsize=font_size)
    ax1.set_ylabel("SHAP Value")
    ax1.tick_params(axis='x', labelrotation=30)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.grid(False)

    ale(
        X=X,
        model=best_xgb_model,
        feature=[feat],
        include_CI=True,
        grid_size=20,
        fig=fig,
        ax=ax2
    )
    ax2.set_title("")
    ax2.axhline(y=0, color="red", linestyle="--", linewidth=1, zorder=0)
    ax2.set_xlabel(feat, fontsize=font_size)
    ax2.set_ylabel("ALE Value")
    ax2.tick_params(axis='x', labelrotation=30)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.grid(False)

    handles, labels = ax2.get_legend_handles_labels()
    if handles:
        ax2.legend(handles, labels, loc="upper left", frameon=False)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

### Spatial Cross Validation Analysis

#### Feature Rank Stability

In [ ]:
def make_spatial_groups_from_df(df, lat_col="Latitude", lon_col="Longitude", n_groups=8, rs=42, min_size=20):
    coords = df[[lat_col, lon_col]].to_numpy()
    for attempt in range(10):
        km = KMeans(n_clusters=n_groups, random_state=rs + attempt, n_init="auto")
        g = km.fit_predict(coords)
        sizes = pd.Series(g).value_counts()
        if sizes.min() >= min_size:
            return g
    return KMeans(n_clusters=n_groups, random_state=rs, n_init="auto").fit_predict(coords)

groups = make_spatial_groups_from_df(df, lat_col="Latitude", lon_col="Longitude", n_groups=16)

feature_cols = list(X.columns)
X_np = X.to_numpy()
y_np = Y.to_numpy()
g_np = groups

# -------------------------
# 1) Spatial CV with fixed hyperparams (refit per fold)
# -------------------------
best_params = {
    "n_estimators": 400, "max_depth": 3, "learning_rate": 0.20,
    "subsample": 1.0, "colsample_bytree": 0.60,
    "min_child_weight": 3, "reg_alpha": 0.001, "reg_lambda": 10.0,
    "objective": "count:poisson", "tree_method": "hist", "random_state": 42
}

gkf = GroupKFold(n_splits=len(np.unique(g_np)))
r2s, maes, rmses, fold_models, fold_idxs = [], [], [], [], []

for (tr, te) in gkf.split(X_np, y_np, groups=g_np):
    mdl = XGBRegressor(**best_params)
    mdl.fit(X_np[tr], y_np[tr])
    pred = mdl.predict(X_np[te])
    r2s.append(r2_score(y_np[te], pred))
    maes.append(mean_absolute_error(y_np[te], pred))
    rmses.append(root_mean_squared_error(y_np[te], pred))
    fold_models.append(mdl)
    fold_idxs.append({"train_idx": tr, "test_idx": te})

print(f"SCV R2 mean±sd: {np.mean(r2s):.3f} ± {np.std(r2s):.3f}")
print(f"SCV MAE mean±sd: {np.mean(maes):.3f} ± {np.std(maes):.3f}")
print(f"SCV RMSE mean±sd: {np.mean(rmses):.3f} ± {np.std(rmses):.3f}")

# -------------------------
# 2) SHAP rank stability across folds (TEST-based)
# -------------------------
import numpy as np
import pandas as pd
import shap
from scipy.stats import spearmanr, rankdata

rng_global = np.random.default_rng(0)

def mean_abs_shap_on_TEST(mdl, Xtr_np, Xte_np, nsamples_bg=2000, nsamples_eval=None, seed=0):
    """
    Compute mean |SHAP| on the TEST subset for a trained fold model.
    Background is drawn from TRAIN rows for TreeExplainer stability.
    """
    rng = np.random.default_rng(seed)

    # background from TRAIN
    bg_n = min(nsamples_bg, Xtr_np.shape[0])
    bg_ix = rng.choice(Xtr_np.shape[0], size=bg_n, replace=False)
    Xbg = Xtr_np[bg_ix]

    # evaluation on TEST
    if nsamples_eval is None or Xte_np.shape[0] <= (nsamples_eval or Xte_np.shape[0]):
        Xev = Xte_np
    else:
        ev_ix = rng.choice(Xte_np.shape[0], size=nsamples_eval, replace=False)
        Xev = Xte_np[ev_ix]

    expl = shap.TreeExplainer(mdl, data=Xbg)
    sv = expl.shap_values(Xev)   # (n_eval, n_features)
    return np.abs(sv).mean(axis=0)

def shap_rank_TEST(mdl, Xtr_np, Xte_np, feature_names, nsamples_bg=2000, nsamples_eval=None, seed=0):
    """
    Return a DataFrame with TEST-based ranks for one fold (1 = most important).
    """
    mean_abs = mean_abs_shap_on_TEST(mdl, Xtr_np, Xte_np,
                                     nsamples_bg=nsamples_bg,
                                     nsamples_eval=nsamples_eval,
                                     seed=seed)
    # ranks: 1 = most important (descending by mean_abs)
    r = rankdata(-mean_abs, method='average')
    order = np.argsort(mean_abs)[::-1]
    return pd.DataFrame({
        "feature":       np.array(feature_names)[order],
        "rank":          r[order],
        "mean_abs_shap": mean_abs[order]
    })

# build per-fold TEST-based ranks
X_np = X.to_numpy()
rank_dfs = []
for fidx, (m, idxs) in enumerate(zip(fold_models, fold_idxs)):
    tr_idx = idxs["train_idx"]
    te_idx = idxs["test_idx"]   # use TEST portion for SHAP ranks
    df_rank = shap_rank_TEST(m,
                             X_np[tr_idx],  # background
                             X_np[te_idx],  # evaluation (TEST)
                             feature_cols,
                             nsamples_bg=2000,
                             nsamples_eval=None,
                             seed=42 + fidx)
    rank_dfs.append(df_rank)

# Spearman rank correlation across folds (all features + top-k by median rank)
features_all = rank_dfs[0]["feature"].tolist()
R = np.vstack([
    df_r.set_index("feature").reindex(features_all)["rank"].to_numpy()
    for df_r in rank_dfs
])  # shape: (n_folds, n_features)

def pairwise_spearman_mean(Rsub):
    cors = []
    for i in range(Rsub.shape[0]):
        for j in range(i+1, Rsub.shape[0]):
            cors.append(spearmanr(Rsub[i], Rsub[j]).statistic)
    return float(np.mean(cors)), float(np.std(cors))

rho_all_mean, rho_all_sd = pairwise_spearman_mean(R)

topn = min(10, R.shape[1])
med = np.median(R, axis=0)         # per-feature median rank across folds
keep = np.argsort(med)[:topn]
rho_top_mean, rho_top_sd = pairwise_spearman_mean(R[:, keep])

print(f"SHAP rank stability (ALL features): ρ={rho_all_mean:.3f} ± {rho_all_sd:.3f}")
print(f"SHAP rank stability (TOP-{topn}): ρ={rho_top_mean:.3f} ± {rho_top_sd:.3f}")

#### SCV Clusters Map

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
import geopandas as gpd
import contextily as ctx

groups = make_spatial_groups_from_df(df, lat_col="Latitude", lon_col="Longitude", n_groups=16)
groups = groups + 1

df_map = df.copy()
df_map["cluster_id"] = groups

cluster_ids = np.sort(np.unique(groups))

custom_colors = [
    "#e6194b", "#3cb44b", "#ffe119", "#4363d8", "#f58231", "#911eb4",
    "#46f0f0", "#f032e6", "#bcf60c", "#fabebe", "#008080", "#e6beff",
    "#9a6324", "#fffac8", "#800000", "#0048BA"
]
cluster_colors = {cid: custom_colors[i % len(custom_colors)] for i, cid in enumerate(cluster_ids)}

lat_col = "Latitude"
lon_col = "Longitude"
group_col = "cluster_id"
point_size = 24
alpha = 0.85
figsize = (9, 7)
BASEMAP_CHOICE = "positron"

BASEMAPS = {
    "positron":  ctx.providers.CartoDB.Positron,
    "voyager":   ctx.providers.CartoDB.Voyager,
    "darkmatter":ctx.providers.CartoDB.DarkMatter,
    "osm":       ctx.providers.OpenStreetMap.Mapnik,
    "satellite": ctx.providers.Esri.WorldImagery,
}
basemap_src = BASEMAPS.get(BASEMAP_CHOICE, ctx.providers.CartoDB.Positron)

required = [lat_col, lon_col, group_col]
missing = [c for c in required if c not in df_map.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

data = df_map.dropna(subset=[lat_col, lon_col, group_col]).copy()
if data.empty:
    raise ValueError("No valid rows after dropping NAs in lat/lon/cluster.")

gdf = gpd.GeoDataFrame(
    data,
    geometry=gpd.points_from_xy(data[lon_col], data[lat_col]),
    crs="EPSG:4326",
).to_crs(epsg=3857)

groups = np.sort(gdf[group_col].unique())
color_map = {g: cluster_colors[g] for g in groups}

fig, ax = plt.subplots(figsize=figsize)

legend_handles = []
for g in groups:
    sub = gdf[gdf[group_col] == g]
    ax.scatter(
        sub.geometry.x.values,
        sub.geometry.y.values,
        s=point_size,
        c=[color_map[g]],
        alpha=alpha,
        edgecolors="black",
        linewidths=0.4,
        label=f"Cluster {g}",
        zorder=3,
    )
    legend_handles.append(
        Line2D([0], [0], marker='o', linestyle='',
               markerfacecolor=color_map[g], markeredgecolor='black',
               markeredgewidth=0.4, markersize=np.sqrt(point_size),
               label=f"Cluster {g}")
    )

minx, miny, maxx, maxy = gdf.total_bounds
pad_x = (maxx - minx) * 0.05 or 1
pad_y = (maxy - miny) * 0.05 or 1
ax.set_xlim(minx - pad_x, maxx + pad_x)
ax.set_ylim(miny - pad_y, maxy + pad_y)

ctx.add_basemap(ax, source=basemap_src, crs=gdf.crs.to_string(), attribution_size=5)

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_aspect("equal", adjustable="box")
ax.grid(False)
ax.legend(handles=legend_handles, title="Clusters",
          frameon=True, fontsize=9, title_fontsize=10, loc="upper right")

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

#### SCV Utilities

In [ ]:
def global_quantile_bins(X_df, feature_name, K=25, q_lo=0.01, q_hi=0.99):
    x = X_df[feature_name].to_numpy()
    lo, hi = np.quantile(x, [q_lo, q_hi])
    x_clip = np.clip(x, lo, hi)
    qs = np.linspace(0, 1, K+1)
    cuts = np.quantile(x_clip, qs)
    cuts[0], cuts[-1] = lo, hi
    mids = 0.5*(cuts[:-1] + cuts[1:])
    return cuts, mids

def compute_fold_shap_values(model, X_bg, X_eval, nsamples_bg=2000, nsamples_eval=4000, seed=0):
    rng = np.random.default_rng(seed)
    if X_bg.shape[0] > nsamples_bg:
        bg_idx = rng.choice(X_bg.shape[0], size=nsamples_bg, replace=False)
        X_bg_use = X_bg[bg_idx]
    else:
        X_bg_use = X_bg
    expl = shap.TreeExplainer(model, data=X_bg_use)

    if X_eval.shape[0] > nsamples_eval:
        ev_idx = rng.choice(X_eval.shape[0], size=nsamples_eval, replace=False)
        X_ev_use = X_eval[ev_idx]
    else:
        X_ev_use = X_eval

    sv = expl.shap_values(X_ev_use)
    return sv, X_ev_use

def shap_trend_for_feature(sv, X_eval, fi, cuts, mids):
    """
    Returns (cuts [K+1], y_edges [K+1]) ready for ax.step(..., where='post').
    Fills any interior NaN runs between the first and last valid bin (no extrapolation).
    """
    x = X_eval[:, fi]
    lo, hi = cuts[0], cuts[-1]
    x_clip = np.clip(x, lo, hi)
    shap_vals = sv[:, fi]

    K = len(mids)
    ybin = np.full(K, np.nan)
    for k in range(K):
        mask = (x_clip >= cuts[k]) & (x_clip < cuts[k+1])
        if np.any(mask):
            ybin[k] = shap_vals[mask].mean()

    # interpolate across the whole span between first and last valid bin
    ok = ~np.isnan(ybin)
    if ok.sum() >= 2:
        idx = np.where(ok)[0]
        L, R = idx[0], idx[-1]
        x_full = np.arange(K)
        y_interp = np.full(K, np.nan)
        y_interp[L:R+1] = np.interp(x_full[L:R+1], idx, ybin[ok])
        ybin = y_interp
    elif ok.sum() == 1:
        # keep the single valid bin, rest stay NaN
        pass
    else:
        # no support at all
        return cuts, np.full(K+1, np.nan)

    # mean-center ignoring NaNs (only interior span has numbers)
    m = np.nanmean(ybin)
    ybin = ybin - m

    # edge-aligned values; last edge uses last valid value if present
    last_valid = np.where(~np.isnan(ybin))[0]
    last_val = ybin[last_valid[-1]] if len(last_valid) else np.nan
    y_edges = np.r_[ybin, last_val]
    return cuts, y_edges

def ale_1d_on_test_with_cuts(model, X_test, fi, cuts):
    x = X_test[:, fi]
    lo, hi = cuts[0], cuts[-1]
    x_clip = np.clip(x, lo, hi)

    K = len(cuts) - 1
    contrib = np.full(K, np.nan, dtype=float)
    for k in range(K):
        mask = (x_clip >= cuts[k]) & (x_clip < cuts[k+1])
        if not np.any(mask): continue
        Xl = X_test.copy(); Xh = X_test.copy()
        Xl[mask, fi] = cuts[k]
        Xh[mask, fi] = cuts[k+1]
        diffs = model.predict(Xh[mask]) - model.predict(Xl[mask])
        contrib[k] = diffs.mean()

    ok = ~np.isnan(contrib)
    if ok.sum() >= 2:
        idx = np.where(ok)[0]; L, R = idx[0], idx[-1]
        x_full = np.arange(K)
        c_interp = np.full(K, np.nan)
        c_interp[L:R+1] = np.interp(x_full[L:R+1], idx, contrib[ok])
        contrib = c_interp
    elif ok.sum() == 0:
        return cuts, np.full(K+1, np.nan)

    ale = np.full(K, np.nan)
    s = np.where(~np.isnan(contrib))[0][0]
    e = np.where(~np.isnan(contrib))[0][-1]
    ale[s:e+1] = np.cumsum(contrib[s:e+1])
    ale = ale - np.nanmean(ale)

    last_valid = np.where(~np.isnan(ale))[0]
    last_val = ale[last_valid[-1]] if len(last_valid) else np.nan
    y_edges = np.r_[ale, last_val]
    return cuts, y_edges

def interpolate_to_union_grid(curves):
    """
    curves: list of (x, y). Returns (xu, Yinterp) where xu is the union grid,
    Yinterp is (n_curves, len(xu)) after interpolation.
    """
    xs = [np.asarray(c[0]) for c in curves]
    ys = [np.asarray(c[1]) for c in curves]

    # Fast path: identical grids across folds (true here because all use `cuts`)
    if all(np.array_equal(x, xs[0]) for x in xs):
        xu = xs[0]
        Y  = np.vstack(ys)
        return xu, Y

    # General path: build union grid and safely interpolate
    xu = np.unique(np.concatenate(xs))
    Ylist = []
    for x, y in curves:
        x_unique, idx = np.unique(x, return_index=True)  # dedupe edges
        y_unique = y[idx]
        Ylist.append(np.interp(xu, x_unique, y_unique))
    Y = np.vstack(Ylist)
    return xu, Y

def rank_topN_by_mean_abs_shap(
    fold_models, fold_idxs, X_df, feature_cols,
    nsamples_bg=2000, nsamples_eval=4000, seed=0
):
    """
    Compute cross-fold mean |SHAP| ranking using TEST rows per fold
    (background from TRAIN rows). Returns (features_sorted, mean_abs_scores_sorted).
    """
    rng = np.random.default_rng(seed)
    M = np.zeros(len(feature_cols), dtype=float)
    C = 0

    X_np = X_df.to_numpy()

    for m, idxs in zip(fold_models, fold_idxs):
        tr = idxs["train_idx"]
        te = idxs["test_idx"]

        Xtr = X_np[tr]
        Xte = X_np[te]

        # background: TRAIN (subsampled)
        bg_n  = min(nsamples_bg, Xtr.shape[0])
        bg_ix = rng.choice(Xtr.shape[0], size=bg_n, replace=False)
        Xbg   = Xtr[bg_ix]

        # evaluation: TEST (subsampled if large)
        ev_n  = min(nsamples_eval, Xte.shape[0])
        ev_ix = rng.choice(Xte.shape[0], size=ev_n, replace=False)
        Xev   = Xte[ev_ix]

        expl = shap.TreeExplainer(m, data=Xbg)
        sv   = expl.shap_values(Xev)  # <<--- TEST rows only

        M += np.abs(sv).mean(axis=0)
        C += 1

    mean_abs = M / max(C, 1)
    order    = np.argsort(mean_abs)[::-1]
    feats    = [feature_cols[i] for i in order]
    scores   = mean_abs[order]
    return feats, scores

#### SHAP and ALE across SCV

In [ ]:
from itertools import cycle
import matplotlib.pyplot as plt
import numpy as np

def plot_topN_features_shap_ale(
    topN,
    fold_models, fold_idxs,
    X_df, feature_cols,
    K_bins=25, q_lo=0.01, q_hi=0.99,
    nsamples_bg=2000, nsamples_eval=4000,
    mean_line_width=2, figsize=(11, 4.8),
    random_seed=0,
    cluster_colors=None
):
    ranked_feats, _ = rank_topN_by_mean_abs_shap(
        fold_models, fold_idxs, X_df, feature_cols, nsamples_bg=4000, seed=random_seed
    )
    feats = ranked_feats[:topN]
    X_np = X_df.to_numpy()

    colors = list(cluster_colors.values()) if cluster_colors else plt.cm.tab20.colors

    for fname in feats:
        if fname not in feature_cols:
            print(f"[skip] Feature not found: {fname}")
            continue
        fi = feature_cols.index(fname)
        cuts, mids = global_quantile_bins(X_df, fname, K=K_bins, q_lo=q_lo, q_hi=q_hi)

        shap_curves = []
        for m, idxs in zip(fold_models, fold_idxs):
            tr, te = idxs["train_idx"], idxs["test_idx"]
            sv, X_eval = compute_fold_shap_values(
                m, X_np[tr], X_np[te],
                nsamples_bg=nsamples_bg, nsamples_eval=nsamples_eval, seed=random_seed
            )
            x_line, y_line = shap_trend_for_feature(sv, X_eval, fi, cuts, mids)
            shap_curves.append((x_line, y_line))
        xu_shap, Y_shap = interpolate_to_union_grid(shap_curves)
        mean_shap = np.nanmean(Y_shap, axis=0)
        min_shap = np.nanmin(Y_shap, axis=0)
        max_shap = np.nanmax(Y_shap, axis=0)

        cuts, mids = global_quantile_bins(X, fname, K=K_bins, q_lo=q_lo, q_hi=q_hi)

        ale_curves = []
        for m, idxs in zip(fold_models, fold_idxs):
            te = idxs["test_idx"]
            x_ale, y_ale = ale_1d_on_test_with_cuts(m, X_np[te], fi, cuts)
            ale_curves.append((x_ale, y_ale))
        xu_ale, Y_ale = interpolate_to_union_grid(ale_curves) 
        mean_ale = np.nanmean(Y_ale, axis=0)
        min_ale = np.nanmin(Y_ale, axis=0)
        max_ale = np.nanmax(Y_ale, axis=0)

        fig, axes = plt.subplots(1, 2, figsize=figsize, constrained_layout=True)
        axL, axR = axes

        color_iter = cycle(colors)
        for y_line, c in zip(Y_shap, color_iter):
            axL.plot(xu_shap, y_line, lw=0.8, color=c, zorder=3)
        axL.fill_between(xu_shap, min_shap, max_shap, color="lightgray", zorder=1, alpha=0.35)
        axL.axhline(y=0, color="red", linestyle="--", linewidth=1, zorder=2)
        axL.set_xlabel(fname, fontsize=9)
        axL.set_ylabel("Stacked SHAP")
        axL.grid(True, alpha=0.25)
        axL.set_facecolor("white")

        color_iter = cycle(colors)
        for y_line, c in zip(Y_ale, color_iter):
            axR.plot(xu_ale, y_line, lw=0.8, color=c, zorder=3)
        axR.fill_between(xu_ale, min_ale, max_ale, color="lightgray", zorder=1, alpha=0.35)
        axR.axhline(y=0, color="red", linestyle="--", linewidth=1, zorder=2)
        axR.set_xlabel(fname, fontsize=9)
        axR.set_ylabel("Stacked ALE")
        axR.set_xlim(axL.get_xlim())
        axR.grid(True, alpha=0.25)
        axR.set_facecolor("white")

        plt.show()

In [ ]:
feature_cols = list(X.columns)
plot_topN_features_shap_ale(
    topN=20,
    fold_models=fold_models,
    fold_idxs=fold_idxs,
    X_df=X,
    feature_cols=feature_cols,
    K_bins=25, q_lo=0.01, q_hi=0.99,
    nsamples_bg=2000, nsamples_eval=4000,
    mean_line_width=2, figsize=(7, 3),
    random_seed=0
)

#### SHAP + ALE CityWide and Across Folds

In [ ]:
def fill_between_visible_lines(ax, x, Y, color="lightgray", alpha=0.35, zorder=1):
    """
    Shade exactly between the polylines drawn by ax.plot(x, y_i).
    - Uses linear interpolation onto a dense grid (~one sample per screen pixel).
    - No extrapolation outside each fold's support.
    - Splits into contiguous supported runs so we never bridge gaps.
    """
    x = np.asarray(x, float)
    Y = np.asarray(Y, float)

    # Dense x grid: ~one sample per pixel across the current x-limits
    xmin, xmax = ax.get_xlim()
    npx = max(512, int(ax.bbox.width))  # at least 512 samples for safety
    xd = np.linspace(xmin, xmax, npx)

    # Interpolate each line onto xd only within its own support
    Yd = np.full((Y.shape[0], xd.size), np.nan)
    for i, y in enumerate(Y):
        ok = np.isfinite(y)
        if ok.sum() >= 2:
            xi, yi = x[ok], y[ok]
            left, right = xi[0], xi[-1]
            span = (xd >= left) & (xd <= right)
            Yd[i, span] = np.interp(xd[span], xi, yi)

    # Where at least one line exists on the dense grid
    support = np.any(np.isfinite(Yd), axis=0)
    if not np.any(support):
        return

    # Break into contiguous runs of support to avoid diagonal bridging
    edges = np.flatnonzero(np.diff(np.r_[False, support, False]))
    runs = edges.reshape(-1, 2)  # [[s1,e1], [s2,e2], ...]

    for s, e in runs:
        Ys = Yd[:, s:e]
        xs = xd[s:e]
        ymin = np.nanmin(Ys, axis=0)
        ymax = np.nanmax(Ys, axis=0)
        ax.fill_between(xs, ymin, ymax, color=color, alpha=alpha, zorder=zorder)


def fill_between_visible_lines_from_artists(ax, lines, color="lightgray", alpha=0.35, zorder=1, samples=None):
    """
    Shade the envelope that exactly hugs the polylines currently drawn on `ax`.
    - Respects NaN breaks and does no extrapolation beyond each contiguous segment.
    - Works with any set of Line2D artists created via ax.plot(...).
    """

    # dense grid ~ one sample per pixel across current xlim
    xmin, xmax = ax.get_xlim()
    if samples is None:
        # use the axes pixel width as density (fallback to 1024 if bbox unavailable)
        try:
            samples = max(512, int(ax.bbox.width))
        except Exception:
            samples = 1024
    xd = np.linspace(xmin, xmax, samples)

    # interpolate each visible line onto xd, per contiguous finite segment
    Yd = np.full((len(lines), xd.size), np.nan)
    for i, ln in enumerate(lines):
        xi = np.asarray(ln.get_xdata(orig=True), dtype=float)
        yi = np.asarray(ln.get_ydata(orig=True), dtype=float)
        finite = np.isfinite(xi) & np.isfinite(yi)
        if not finite.any():
            continue
        edges = np.flatnonzero(np.diff(np.r_[False, finite, False])).reshape(-1, 2)
        for s, e in edges:
            xs = xi[s:e]; ys = yi[s:e]
            if xs.size < 2:
                continue
            span = (xd >= xs[0]) & (xd <= xs[-1])  # no extrapolation
            Yd[i, span] = np.interp(xd[span], xs, ys)

    # columns where at least one line exists
    support = np.any(np.isfinite(Yd), axis=0)
    if not np.any(support):
        return

    # fill per contiguous supported run (prevents diagonal bridging)
    runs = np.flatnonzero(np.diff(np.r_[False, support, False])).reshape(-1, 2)
    for s, e in runs:
        Ys = Yd[:, s:e]
        xs = xd[s:e]
        ymin = np.nanmin(Ys, axis=0)
        ymax = np.nanmax(Ys, axis=0)
        ax.fill_between(xs, ymin, ymax, color=color, alpha=alpha, zorder=zorder)

In [ ]:
def plot_shap_ale_combined(
    best_xgb_model,
    X, Y,
    fold_models, fold_idxs,
    feature_cols,
    cluster_colors,
    top_features,
    K_bins=30, q_lo=0, q_hi=1,
    nsamples_bg=2000, nsamples_eval=4000,
    random_seed=0
):
    explainer = shap.TreeExplainer(best_xgb_model)
    shap_values = explainer(X)

    top_feats = list(top_features)
    top_idx = [X.columns.get_loc(f) for f in top_feats]

    colors = list(cluster_colors.values()) if cluster_colors else plt.cm.tab20.colors

    for feat, idx in zip(top_feats, top_idx):
        x = X.iloc[:, idx]
        ys = shap_values.values[:, idx]
        fig, axes = plt.subplots(2, 2, figsize=(9, 6), constrained_layout=True)
        ax1, ax2 = axes[0]
        ax3, ax4 = axes[1]

        print(f"SHAP and ALE Interpretation for {feat}")

        # --- top-left: citywide SHAP scatter ---
        ax1.scatter(x, ys, s=10, alpha=0.5, color="steelblue")
        ax1.axhline(y=0, color="red", linestyle="--", linewidth=1, zorder=0)
        ax1.set_ylabel("Citywide SHAP Value")
        ax1.spines['top'].set_visible(False)
        ax1.spines['right'].set_visible(False)
        ax1.grid(False)
        ax1.set_xlabel(feat)

        # --- top-right: citywide ALE (library) ---
        ale(
            X=X,
            model=best_xgb_model,
            feature=[feat],
            include_CI=True,
            grid_size=20,
            fig=fig,
            ax=ax2
        )
        ax2.set_title("")
        ax2.axhline(y=0, color="red", linestyle="--", linewidth=1, zorder=0)
        ax2.set_ylabel("Citywide ALE Value")
        ax2.spines['top'].set_visible(False)
        ax2.spines['right'].set_visible(False)
        ax2.grid(False)
        handles, labels = ax2.get_legend_handles_labels()
        if handles:
            ax2.legend(handles, labels, loc="upper left", frameon=False)

        # --- shared grid for stacked plots ---
        fname = feat
        fi = feature_cols.index(fname)
        cuts, mids = global_quantile_bins(X, fname, K=K_bins, q_lo=q_lo, q_hi=q_hi)
        X_np = X.to_numpy()

        # --- bottom-left: stacked SHAP (SCV) ---
        shap_curves = []
        for m, idxs in zip(fold_models, fold_idxs):
            tr, te = idxs["train_idx"], idxs["test_idx"]
            sv, X_eval = compute_fold_shap_values(
                m, X_np[tr], X_np[te],
                nsamples_bg=nsamples_bg, nsamples_eval=nsamples_eval, seed=random_seed
            )
            x_line, y_line = shap_trend_for_feature(sv, X_eval, fi, cuts, mids)  # returns (cuts, y_edges)
            shap_curves.append((x_line, y_line))
        xu_shap, Y_shap = interpolate_to_union_grid(shap_curves)  # fast-path stacks; same edges

        # draw lines
        for y_line, c in zip(Y_shap, cycle(colors)):
            ax3.plot(xu_shap, y_line, lw=0.8, color=c, zorder=3)

        ax3.axhline(y=0, color="red", linestyle="--", linewidth=1, zorder=2)
        ax3.set_xlabel(fname, fontsize=9)
        ax3.set_ylabel("Stacked SHAP Value Across SCV Folds")
        ax3.grid(True, alpha=0.25)
        ax3.set_facecolor("white")
        ax3.set_xlim(xu_shap[0], xu_shap[-1])

        # gray band that hugs the actually drawn SHAP lines
        shap_lines = list(ax3.get_lines())
        fill_between_visible_lines_from_artists(ax3, shap_lines, color="lightgray", alpha=0.35, zorder=1)

        # --- bottom-right: stacked ALE (SCV) ---
        # ensure ALE uses the same global cuts
        ale_curves = []
        for m, idxs in zip(fold_models, fold_idxs):
            te = idxs["test_idx"]
            x_ale, y_ale = ale_1d_on_test_with_cuts(m, X_np[te], fi, cuts)  # returns (cuts, y_edges)
            ale_curves.append((x_ale, y_ale))
        xu_ale, Y_ale = interpolate_to_union_grid(ale_curves)

        # draw lines
        for y_line, c in zip(Y_ale, cycle(colors)):
            ax4.plot(xu_ale, y_line, lw=0.8, color=c, zorder=3)

        ax4.axhline(y=0, color="red", linestyle="--", linewidth=1, zorder=2)
        ax4.set_xlabel(fname, fontsize=9)
        ax4.set_ylabel("Stacked ALE Value Across SCV Folds")
        ax4.grid(True, alpha=0.25)
        ax4.set_facecolor("white")
        ax4.set_xlim(xu_ale[0], xu_ale[-1])

        # gray band that hugs the actually drawn ALE lines
        ale_lines = list(ax4.get_lines())
        fill_between_visible_lines_from_artists(ax4, ale_lines, color="lightgray", alpha=0.35, zorder=1)

        plt.show()

In [ ]:
rank_df = rank_topN_by_mean_abs_shap(
        fold_models, fold_idxs, X, feature_cols, nsamples_bg=4000, seed=0
    )
r_feats, r_scores = rank_df
r_feats = list(r_feats)
r_scores = np.asarray(r_scores, dtype=float)
order = np.argsort(r_scores)[::-1]
top_features = [r_feats[i] for i in order[:22]]

plot_shap_ale_combined(
    best_xgb_model=run_best_model(X, Y),
    X=X,
    Y=Y,
    fold_models=fold_models,
    fold_idxs=fold_idxs,
    feature_cols=list(X.columns),
    cluster_colors=cluster_colors,
    top_features=top_features
)

#### Base + SCV SHAP Ranking

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import textwrap
import pandas as pd

def plot_grouped_topN_shap(
    ax,
    rank_output,                 
    feature_importance_df,
    topN=19,
    feat_col="Feature",
    shap_col="Mean SHAP Value",
    label_max_len=40,
    annotate=True,
    decimal_places=3,
    title="Top SHAP Features",
    rank_label="Spatial CV mean |SHAP|",
    df_label="Fixed Split mean |SHAP|"
):
    
    r_feats, r_scores = rank_output
    r_feats  = list(r_feats)
    r_scores = np.asarray(r_scores, dtype=float)
    order = np.argsort(r_scores)[::-1]
    r_feats  = [r_feats[i] for i in order]
    r_scores = r_scores[order]

    topN = min(topN, len(r_feats))
    feats_top     = r_feats[:topN]
    r_scores_top  = r_scores[:topN]

    if feat_col not in feature_importance_df.columns or shap_col not in feature_importance_df.columns:
        raise ValueError(f"feature_importance_df must have columns '{feat_col}' and '{shap_col}'")
    df_map = dict(zip(feature_importance_df[feat_col], feature_importance_df[shap_col]))
    d_scores_top = np.array([df_map.get(f, np.nan) for f in feats_top], dtype=float)
    d_scores_top = np.nan_to_num(d_scores_top, nan=0.0)

    def wrap_label(s, width):
        return "\n".join(textwrap.wrap(s, width=width)) if len(s) > width else s
    wrapped_labels = [wrap_label(s, label_max_len) for s in feats_top]

    y = np.arange(topN)
    bar_h = 0.4
    bars_rank = ax.barh(
        y - bar_h/2, 
        r_scores_top, 
        height=bar_h, 
        alpha=0.9, 
        label=rank_label,
        color="#4C72B0")
    bars_df = ax.barh(
        y + bar_h/2,
        d_scores_top,
        height=bar_h,
        alpha=0.9,
        label=df_label,
        color="#55A868"
    )

    ax.set_yticks(y)
    ax.set_yticklabels(wrapped_labels, fontsize=9)
    ax.set_xlabel("Mean |SHAP|")
    ax.set_title(title)
    ax.legend(loc="lower right")

    ax.invert_yaxis()

    xmax = float(np.max([r_scores_top.max() if len(r_scores_top) else 0.0,
                         d_scores_top.max() if len(d_scores_top) else 0.0]))
    ax.set_xlim(0, xmax * 1.12 if xmax > 0 else 1.0)

    if annotate:
        for b, val in zip(bars_rank, r_scores_top):
            x = b.get_width()
            ax.text(x * 1.01, b.get_y() + b.get_height()/2,
                    f"{val:.{decimal_places}f}", va='center', ha='left', fontsize=8, clip_on=False)
        for b, val in zip(bars_df, d_scores_top):
            x = b.get_width()
            ax.text(x * 1.01, b.get_y() + b.get_height()/2,
                    f"{val:.{decimal_places}f}", va='center', ha='left', fontsize=8, clip_on=False)


fig, ax = plt.subplots(figsize=(10, 8))

rank_df = rank_topN_by_mean_abs_shap(
        fold_models, fold_idxs, X, feature_cols, nsamples_bg=4000, seed=0
    )

plot_grouped_topN_shap(
    ax,
    rank_output=rank_df,
    feature_importance_df=feature_importance_df,
    topN=22,
    title="Top SHAP Features",
    rank_label="Spatial CV mean |SHAP|",
    df_label="Citywide mean |SHAP|" 
)

fig.set_dpi(300)
plt.tight_layout()
plt.show()

### Scenario Analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

POLICY_TO_COLS = {
    "Average wait time capped at 40 seconds":   ["Average Pedestrian Wait Time (s)"],
    "Average vehicle speeds capped at 50 km/h":  ["Average Vehicle Speed (km/h)"],
    "Percentage of drive alone commuters maxed at 65%":  ["Percentage of Drive Alone Commuters"],
    "shops_ge_2":    ["Shop POIs"],
    "leisure_ge_10": ["Leisure POIs"],
}

def apply_policy_in_place(Xs, policy_name):
    if policy_name == "Average wait time capped at 40 seconds" and "Average Pedestrian Wait Time (s)" in Xs.columns:
        Xs["Average Pedestrian Wait Time (s)"] = np.minimum(Xs["Average Pedestrian Wait Time (s)"], 40)
    elif policy_name == "Average vehicle speeds capped at 50 km/h" and "Average Vehicle Speed (km/h)" in Xs.columns:
        Xs["Average Vehicle Speed (km/h)"] = np.minimum(Xs["Average Vehicle Speed (km/h)"], 50)
    elif policy_name == "shops_ge_2" and "Shop POIs" in Xs.columns:
        Xs["Shop POIs"] = np.where(Xs["Shop POIs"] < 2, 2, Xs["Shop POIs"])
    elif policy_name == "leisure_ge_10" and "Leisure POIs" in Xs.columns:
        Xs["Leisure POIs"] = np.where(Xs["Leisure POIs"] < 10, 10, Xs["Leisure POIs"])
    elif policy_name == "Percentage of drive alone commuters maxed at 65%" and "Percentage of Drive Alone Commuters" in Xs.columns:
        Xs["Percentage of Drive Alone Commuters"] = np.where(Xs["Percentage of Drive Alone Commuters"] > 0.65, 65, Xs["Percentage of Drive Alone Commuters"])
    return Xs

def _changed_mask_robust(X_before, X_after, cols, atol=1e-9):
    cols = [c for c in cols if c in X_before.columns and c in X_after.columns]
    if not cols:
        return np.zeros(len(X_before), dtype=bool)
    out = np.zeros(len(X_before), dtype=bool)
    for c in cols:
        a = X_before[c].to_numpy()
        b = X_after[c].to_numpy()
        both_nan = pd.isna(a) & pd.isna(b)
        if pd.api.types.is_numeric_dtype(X_before[c]) and pd.api.types.is_numeric_dtype(X_after[c]):
            diff = np.abs((b - a).astype(float))
            eq = np.isclose(diff, 0.0, atol=atol) | both_nan
        else:
            eq = (a == b) | both_nan
        out |= ~eq
    return out

def _cols_for_policy(policy_name, X):
    return [c for c in POLICY_TO_COLS.get(policy_name, []) if c in X.columns]


def _device_mean_for_cols(X_base, df_device, device_col, cols):
    """
    Returns Series indexed by device: scalar mean baseline value for those cols (avg across cols).
    """
    cols = [c for c in cols if c in X_base.columns]
    if not cols:
        return pd.Series(dtype=float)

    base_vals = X_base[cols].apply(pd.to_numeric, errors="coerce")
    tmp = base_vals.copy()
    tmp["device"] = df_device[device_col].values

    # mean per device per col, then average across cols -> scalar per device
    dev_mean = tmp.groupby("device")[cols].mean(numeric_only=True).mean(axis=1)
    return dev_mean


def _row_avg_abs_change(X_base, X_after, cols):
    """
    Per-row: average absolute change across given cols.
    """
    cols = [c for c in cols if c in X_base.columns and c in X_after.columns]
    if not cols:
        return np.zeros(len(X_base), dtype=float)

    a = X_base[cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    b = X_after[cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)

    row_abs = np.nanmean(np.abs(b - a), axis=1)
    row_abs = np.where(np.isfinite(row_abs), row_abs, 0.0)
    return row_abs


def _row_change_metric_vs_device_mean(
    X_base, X_after, df_device, device_col, cols, mode="percent"
):
    """
    mode:
      - "percent": (avg abs change across cols) / |device mean baseline| * 100
      - "absolute": avg abs change across cols (raw units)
    """
    row_abs = _row_avg_abs_change(X_base, X_after, cols)

    if mode == "absolute":
        return row_abs

    dev_mean = _device_mean_for_cols(X_base, df_device, device_col, cols)
    dev_for_row = df_device[device_col].values
    dev_mean_for_row = dev_mean.reindex(dev_for_row).to_numpy(dtype=float)

    denom = np.where(np.isfinite(dev_mean_for_row) & (np.abs(dev_mean_for_row) > 1e-12),
                     np.abs(dev_mean_for_row),
                     np.nan)

    pct = (row_abs / denom) * 100.0
    pct = np.where(np.isfinite(pct), pct, 0.0)
    return pct

def _compute_policy_device_changes(
    X_base,
    policy_name,
    df_device,
    device_col,
    qualifying_devices,
    absolute_mode=False,
):
    """
    For devices passing gating, compute device-level average "reduction" under THIS policy.

    Returns:
      - changes: np.ndarray (one per qualifying device, sorted desc, NaNs removed)
      - meta: dict with info (policy, mode, n_devices, cols used)

    change definition:
      - percent mode: ((base_mean - after_mean)/|base_mean|)*100
      - absolute mode: (base_mean - after_mean) in column units
    """
    cols = _cols_for_policy(policy_name, X_base)
    if not cols:
        return np.array([]), {"policy": policy_name, "mode": "absolute" if absolute_mode else "percent",
                              "n_devices": 0, "cols": []}

    X_after = X_base.copy()
    apply_policy_in_place(X_after, policy_name)

    devs = sorted(list(qualifying_devices))
    if not devs:
        return np.array([]), {"policy": policy_name, "mode": "absolute" if absolute_mode else "percent",
                              "n_devices": 0, "cols": cols}

    base_vals = X_base[cols].apply(pd.to_numeric, errors="coerce")
    after_vals = X_after[cols].apply(pd.to_numeric, errors="coerce")

    dev_for_row = df_device[device_col].values
    tmp = pd.DataFrame({"device": dev_for_row})
    for c in cols:
        tmp[f"b_{c}"] = base_vals[c].values
        tmp[f"a_{c}"] = after_vals[c].values

    changes = []
    for d in devs:
        sub = tmp[tmp["device"] == d]
        base_mean = np.nanmean([np.nanmean(sub[f"b_{c}"].to_numpy(dtype=float)) for c in cols])
        after_mean = np.nanmean([np.nanmean(sub[f"a_{c}"].to_numpy(dtype=float)) for c in cols])

        if not np.isfinite(base_mean) or np.abs(base_mean) < 1e-12:
            ch = np.nan
        else:
            if absolute_mode:
                ch = (base_mean - after_mean)  # raw units
            else:
                ch = ((base_mean - after_mean) / np.abs(base_mean)) * 100.0  # percent
        changes.append(ch)

    changes = np.array(changes, dtype=float)
    changes = changes[np.isfinite(changes)]
    if changes.size:
        changes = np.sort(changes)[::-1]  # desc
    meta = {
        "policy": policy_name,
        "mode": "absolute" if absolute_mode else "percent",
        "n_devices": int(changes.size),
        "cols": cols,
    }
    return changes, meta

def kpi_run(
    model,
    X,
    policies,
    df_device,
    device_col="DeviceID",
    pop_col="Population within Quarter Mile",
    gating="Union",
    gate_threshold=0.5,
    boots=1000,
    seed=0,
    hist_bins=10,
    practical_limits=None,
):
    """
    Same metrics as before, but the figure is now a dot + box/violin plot
    of device-level %Δ (affected intersections only).
    """
    if df_device is None or device_col not in df_device.columns:
        raise ValueError("df_device with a valid DeviceID column is required.")
    rng = np.random.default_rng(seed)

    per_policy_masks = []
    per_policy_cols  = []
    for p in policies:
        Xp = X.copy()
        apply_policy_in_place(Xp, p)
        cols_p = [c for c in POLICY_TO_COLS.get(p, []) if c in X.columns]
        per_policy_cols.append(cols_p)
        per_policy_masks.append(_changed_mask_robust(X, Xp, cols_p, atol=1e-9))

    cols_all = sorted({c for cols in per_policy_cols for c in cols})

    if len(per_policy_masks) == 0:
        row_gate_mask = np.zeros(len(X), dtype=bool)
    elif gating == "Intersect":
        row_gate_mask = np.logical_and.reduce(per_policy_masks)
    else:
        row_gate_mask = np.logical_or.reduce(per_policy_masks)

    gate_df = pd.DataFrame({
        "device": df_device[device_col].values,
        "row_changed": row_gate_mask
    })
    per_dev = (gate_df.groupby("device", as_index=False)
                     .agg(rows_total=("row_changed", "size"),
                          rows_changed=("row_changed", "sum")))
    per_dev["prop_changed"] = per_dev["rows_changed"] / per_dev["rows_total"].clip(lower=1)
    qualifying_devices = set(per_dev.loc[per_dev["prop_changed"] >= gate_threshold, "device"])

    X_candidate = X.copy()
    for p in policies:
        X_candidate = apply_policy_in_place(X_candidate, p)

    Xs = X.copy()
    mask_apply = df_device[device_col].isin(qualifying_devices).to_numpy()

    ##
    # --- NEW: per-policy practical plots + practical row filter ---
    practical_limits = practical_limits or {}
    practical_device_changes = {}  # policy -> {"values": np.array, "meta": {...}}

    for p in policies:
        if p in practical_limits and _cols_for_policy(p, X):
            x_val = practical_limits[p].get("x", None)
            abs_flag = bool(practical_limits[p].get("absolute", False))
            if x_val is not None:
                vals, meta = _compute_policy_device_changes(
                    X_base=X,
                    policy_name=p,
                    df_device=df_device,
                    device_col=device_col,
                    qualifying_devices=qualifying_devices,
                    absolute_mode=abs_flag,
                )
                practical_device_changes[p] = {"values": vals, "meta": meta, "threshold": x_val}

    # practical row mask: row must pass ALL provided policy practical tests,
    # but if a given policy doesn't change a row, that policy doesn't block it.
    practical_row_mask = np.ones(len(X), dtype=bool)

    for p in policies:
        if p not in practical_limits:
            continue

        cols_p = _cols_for_policy(p, X)
        if not cols_p:
            continue

        x_val = practical_limits[p].get("x", None)
        if x_val is None:
            continue

        abs_flag = bool(practical_limits[p].get("absolute", False))

        # Apply only this policy to measure "change by implementing the policy"
        Xp_only = X.copy()
        apply_policy_in_place(Xp_only, p)

        # rows where this policy actually changes something
        changed_p = _changed_mask_robust(X, Xp_only, cols_p, atol=1e-9)

        mode = "absolute" if abs_flag else "percent"
        metric_p = _row_change_metric_vs_device_mean(
            X_base=X,
            X_after=Xp_only,
            df_device=df_device,
            device_col=device_col,
            cols=cols_p,
            mode=mode,
        )

        pass_p = metric_p < float(x_val)

        # If row not changed by this policy, don't block it.
        practical_row_mask &= (~changed_p) | pass_p
    ##

    for c in cols_all:
        Xs.loc[mask_apply, c] = X_candidate.loc[mask_apply, c]

    # --- NEW: practical scenario applies policy only to rows passing practical_row_mask too ---
    Xs_practical = X.copy()
    mask_apply_practical = mask_apply & practical_row_mask
    for c in cols_all:
        Xs_practical.loc[mask_apply_practical, c] = X_candidate.loc[mask_apply_practical, c]

    yB = model.predict(X.values)
    yS = model.predict(Xs.values)
    delta = (yS - yB) / (yB + 1e-6)

    n_total = len(delta)
    meds = [np.median(delta[rng.integers(0, n_total, n_total)]) for _ in range(boots)]
    lo, hi = np.percentile(meds, [2.5, 97.5])
    city_mean = float(delta.mean() * 100)
    city_median = float(np.median(delta) * 100)
    city_iqr = [float(np.percentile(delta, 25) * 100), float(np.percentile(delta, 75) * 100)]
    sites_improved_pct_row = float((delta > 0).mean() * 100)

    impacted_mask = mask_apply & _changed_mask_robust(X, Xs, cols_all, atol=1e-9)
    tmp = pd.DataFrame({
        "device": df_device[device_col].values,
        "delta": delta,
        "impacted": impacted_mask
    })
    dev_agg = (tmp.groupby("device", as_index=False)
                 .agg(affected=("impacted", "any"),
                      rows_impacted=("impacted", "sum"),
                      rows_total=("impacted", "size"),
                      delta_mean=("delta", "mean")))
    dev_agg["improved"] = dev_agg["delta_mean"] > 0

    if pop_col in X.columns:
        pop_series = pd.to_numeric(X[pop_col], errors="coerce")
    else:
        pop_series = pd.Series(np.nan, index=X.index)
    pop_by_dev = (pd.DataFrame({"device": df_device[device_col].values, "pop": pop_series.values})
                    .groupby("device", as_index=False)
                    .agg(pop=("pop", lambda s: pd.to_numeric(s, errors="coerce").dropna().iloc[0]
                                       if pd.to_numeric(s, errors="coerce").dropna().size > 0 else 0.0)))
    dev_agg = dev_agg.merge(pop_by_dev, on="device", how="left").fillna({"pop": 0.0})

    n_devices_total = int(len(dev_agg))
    n_devices_affected = int(dev_agg["affected"].sum())
    n_devices_improved = int(dev_agg["improved"].sum())
    affected_sites_pct = float(100.0 * n_devices_affected / max(n_devices_total, 1))
    improved_aff_pct = (float(100.0 * (dev_agg.loc[dev_agg["affected"], "improved"].mean()))
                        if n_devices_affected > 0 else None)

    if n_devices_affected > 0:
        d = dev_agg.loc[dev_agg["affected"], "delta_mean"].to_numpy()
        mean_aff = float(d.mean() * 100)
        median_aff = float(np.median(d) * 100)
        iqr_aff = [float(np.percentile(d, 25) * 100), float(np.percentile(d, 75) * 100)]
        device_delta_pct = (d * 100.0)
    else:
        mean_aff = median_aff = improved_aff_pct = None
        iqr_aff = None
        device_delta_pct = np.array([])

    total_pop = float(dev_agg["pop"].sum())
    affected_pop = float(dev_agg.loc[dev_agg["affected"], "pop"].sum())
    people_affected = affected_pop
    people_affected_ratio = (affected_pop / total_pop * 100.0) if total_pop > 0 else None

    histogram_data = []
    if n_devices_affected and n_devices_affected > 0:
        aff = dev_agg.loc[dev_agg["affected"], ["device", "delta_mean"]].copy()
        aff["delta_pct"] = aff["delta_mean"] * 100.0
        aff = aff.sort_values("delta_pct", ascending=True).reset_index(drop=True)
        N = len(aff)
        B = int(hist_bins) if hist_bins and hist_bins > 0 else 10
        edges = np.linspace(0, N, B+1, dtype=int)
        for i in range(B):
            lo_idx, hi_idx = edges[i], edges[i+1]
            if hi_idx <= lo_idx:
                continue
            bin_slice = aff.iloc[lo_idx:hi_idx]
            histogram_data.append({
                "bin": i+1,
                "bin_start_pct": float((lo_idx / N) * 100.0),
                "bin_end_pct": float((hi_idx / N) * 100.0),
                "n_devices": int(len(bin_slice)),
                "mean_delta_pct": float(bin_slice["delta_pct"].mean()),
                "median_delta_pct": float(bin_slice["delta_pct"].median())
            })

    print("-------")
    print("policies:", " + ".join(policies) + (f" ({gating})" if len(policies) > 1 else ""))
    print("gating_mode:", gating, " | gate_threshold:", gate_threshold)
    print("n_devices_total:", n_devices_total)
    print("n_devices_affected:", n_devices_affected)
    print("n_devices_improved:", n_devices_improved)
    print("affected_sites_% (device-level):", affected_sites_pct)
    print("sites_improved_among_affected_% (device-level):", improved_aff_pct)
    print("mean_%Δ_affected_only (device-level):", mean_aff)
    print("median_%Δ_affected_only (device-level):", median_aff)
    print("IQR_%Δ_affected_only (device-level):", iqr_aff)
    print("people_affected:", people_affected)
    print("people_affected_ratio %:", people_affected_ratio)
    print("citywide_mean_%Δ:", city_mean)
    print("median_%Δ:", city_median)
    print("PI95_median_%Δ:", [float(lo * 100), float(hi * 100)])
    print("IQR_%Δ:", city_iqr)
    print("sites_improved_% (row-level):", sites_improved_pct_row)

    # --- NEW: practical_scenario KPI printout (same metrics, different scenario) ---
    yS_pr = model.predict(Xs_practical.values)
    delta_pr = (yS_pr - yB) / (yB + 1e-6)

    n_total_pr = len(delta_pr)
    meds_pr = [np.median(delta_pr[rng.integers(0, n_total_pr, n_total_pr)]) for _ in range(boots)]
    lo_pr, hi_pr = np.percentile(meds_pr, [2.5, 97.5])

    city_mean_pr = float(delta_pr.mean() * 100)
    city_median_pr = float(np.median(delta_pr) * 100)
    city_iqr_pr = [float(np.percentile(delta_pr, 25) * 100), float(np.percentile(delta_pr, 75) * 100)]
    sites_improved_pct_row_pr = float((delta_pr > 0).mean() * 100)

    impacted_mask_pr = mask_apply_practical & _changed_mask_robust(X, Xs_practical, cols_all, atol=1e-9)
    tmp_pr = pd.DataFrame({
        "device": df_device[device_col].values,
        "delta": delta_pr,
        "impacted": impacted_mask_pr
    })
    dev_agg_pr = (tmp_pr.groupby("device", as_index=False)
                    .agg(affected=("impacted", "any"),
                         rows_impacted=("impacted", "sum"),
                         rows_total=("impacted", "size"),
                         delta_mean=("delta", "mean")))
    dev_agg_pr["improved"] = dev_agg_pr["delta_mean"] > 0

    dev_agg_pr = dev_agg_pr.merge(pop_by_dev, on="device", how="left").fillna({"pop": 0.0})

    n_devices_total_pr = int(len(dev_agg_pr))
    n_devices_affected_pr = int(dev_agg_pr["affected"].sum())
    n_devices_improved_pr = int(dev_agg_pr["improved"].sum())
    affected_sites_pct_pr = float(100.0 * n_devices_affected_pr / max(n_devices_total_pr, 1))
    improved_aff_pct_pr = (float(100.0 * (dev_agg_pr.loc[dev_agg_pr["affected"], "improved"].mean()))
                           if n_devices_affected_pr > 0 else None)

    if n_devices_affected_pr > 0:
        dpr = dev_agg_pr.loc[dev_agg_pr["affected"], "delta_mean"].to_numpy()
        mean_aff_pr = float(dpr.mean() * 100)
        median_aff_pr = float(np.median(dpr) * 100)
        iqr_aff_pr = [float(np.percentile(dpr, 25) * 100), float(np.percentile(dpr, 75) * 100)]
    else:
        mean_aff_pr = median_aff_pr = None
        iqr_aff_pr = None

    total_pop_pr = float(dev_agg_pr["pop"].sum())
    affected_pop_pr = float(dev_agg_pr.loc[dev_agg_pr["affected"], "pop"].sum())
    people_affected_pr = affected_pop_pr
    people_affected_ratio_pr = (affected_pop_pr / total_pop_pr * 100.0) if total_pop_pr > 0 else None

    print("-------")
    print("practical_scenario (policy-specific thresholds):", practical_limits)
    print("policies:", " + ".join(policies) + (f" ({gating})" if len(policies) > 1 else ""))
    print("gating_mode:", gating, " | gate_threshold:", gate_threshold)
    print("n_devices_total:", n_devices_total_pr)
    print("n_devices_affected:", n_devices_affected_pr)
    print("n_devices_improved:", n_devices_improved_pr)
    print("affected_sites_% (device-level):", affected_sites_pct_pr)
    print("sites_improved_among_affected_% (device-level):", improved_aff_pct_pr)
    print("mean_%Δ_affected_only (device-level):", mean_aff_pr)
    print("median_%Δ_affected_only (device-level):", median_aff_pr)
    print("IQR_%Δ_affected_only (device-level):", iqr_aff_pr)
    print("people_affected:", people_affected_pr)
    print("people_affected_ratio %:", people_affected_ratio_pr)
    print("citywide_mean_%Δ:", city_mean_pr)
    print("median_%Δ:", city_median_pr)
    print("PI95_median_%Δ:", [float(lo_pr * 100), float(hi_pr * 100)])
    print("IQR_%Δ:", city_iqr_pr)
    print("sites_improved_% (row-level):", sites_improved_pct_row_pr)

    return practical_device_changes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_practical_device_changes(
    values,
    *,
    unit="%",                     # e.g. "%", "mph", "miles per hour"
    y_label=None,                 # e.g. "Average reduction"
    title=None,
    value_mode="auto",            # "auto" | "percent" | "absolute" (only used for default y_label if y_label=None)
    source_mode=None,             # meta["mode"] when value_mode="auto"
    extra_lines=None,             # list of line numeric values, e.g. [15, 10]
    extra_line_labels=None,       # list of labels for those lines (optional)
    extra_line_colors=None,       # list of colors for those lines, e.g. ["orange","red"]
    show_mean=True,
    mean_label="Mean Reduction Across Intersections",
    bar_width=0.35,
    bar_color="gray",
    mean_color="blue",
    line_width=1.5,
):
    """
    values: 1D array-like of device-level changes (already computed).
    unit: string appended in legend labels for horizontal lines (e.g., '%' or 'mph').
    extra_lines: horizontal reference lines (numbers)
    extra_line_colors: same length as extra_lines; if shorter, remaining colors cycle.
    extra_line_labels: same length as extra_lines; if None, auto-labels like "Line 1".
    """
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return

    # Determine mode if user wants auto behavior (only affects default y-label)
    if value_mode == "auto":
        if source_mode is None:
            raise ValueError("If value_mode='auto', you must pass source_mode (e.g., meta['mode']).")
        mode = source_mode
    else:
        mode = value_mode

    vals = np.sort(vals)[::-1]

    plt.figure(figsize=(8, 3.5))
    x = np.arange(len(vals))

    # Bars: gray
    plt.bar(x, vals, width=bar_width, color=bar_color)
    plt.xticks([])

    # Default y label if not provided
    if y_label is None:
        y_label = "Change (units)" if mode == "absolute" else "Change (%)"
    plt.ylabel(y_label)

    if title is not None:
        plt.title(title)

    plt.xlabel("Intersections")

    handles = []
    labels = []

    # Mean line: blue, in legend
    if show_mean:
        m = float(np.mean(vals))
        h = plt.axhline(m, color=mean_color, linestyle="-", linewidth=line_width)
        handles.append(h)
        labels.append(f"{mean_label}".rstrip())

    # Extra lines: with provided colors, all in legend
    extra_lines = extra_lines or []
    if len(extra_lines) > 0:
        if extra_line_colors is None or len(extra_line_colors) == 0:
            extra_line_colors = ["orange", "red"]  # default cycling
        if extra_line_labels is None:
            extra_line_labels = [f"Line {i+1}" for i in range(len(extra_lines))]

        for i, v in enumerate(extra_lines):
            c = extra_line_colors[i % len(extra_line_colors)]
            lab = extra_line_labels[i] if i < len(extra_line_labels) else f"Line {i+1}"
            h = plt.axhline(float(v), color=c, linestyle="--", linewidth=0.8)
            handles.append(h)
            labels.append(f"{lab}".rstrip())

    if handles:
        plt.legend(handles, labels, loc="best", frameon=True)

    plt.tight_layout()
    plt.show()

In [ ]:
best_xgb_model = run_best_model(X, Y)

practical_limits = {
  "Average wait time capped at 40 seconds": {"x": 15,  "absolute": False},
}

practical_device_changes = kpi_run(best_xgb_model, X, 
    ["Average wait time capped at 40 seconds"], df_device=df, practical_limits=practical_limits)

p = "Average wait time capped at 40 seconds"
vals = practical_device_changes[p]["values"]
meta = practical_device_changes[p]["meta"]

plot_practical_device_changes(
    vals,
    unit="%",
    y_label="Average pedestrian wait time reduction (%)",
    # title="Intersection-Level Average Reduction in Pedestrian Wait Time Under a 40-Second Cap Policy",
    value_mode="auto",
    source_mode=meta["mode"],
    extra_lines=[15, 30],
    extra_line_labels=["15% max reduction (High Feasibility)", 
                       "30% max reduction (Moderate Feasibility)"],
    extra_line_colors=["orange", "red"],
    show_mean=True,
)

In [ ]:
practical_limits = {
  "Average wait time capped at 40 seconds": {"x": 30,  "absolute": False},
}

practical_device_changes = kpi_run(best_xgb_model, X, 
    ["Average wait time capped at 40 seconds"], df_device=df, practical_limits=practical_limits)

In [ ]:
practical_limits = {
  "Average vehicle speeds capped at 50 km/h": {"x": 10,  "absolute": True},
}

practical_device_changes = kpi_run(best_xgb_model, X, 
    ["Average vehicle speeds capped at 50 km/h"], df_device=df, practical_limits=practical_limits)

p = "Average vehicle speeds capped at 50 km/h"
vals = practical_device_changes[p]["values"]
meta = practical_device_changes[p]["meta"]

plot_practical_device_changes(
    vals,
    unit="absolute",
    y_label="Average speed reduction (km/h)",
    # title="Intersection-Level Average Reduction in Vehicle Speeds Under a 50 km/h Cap Policy",
    value_mode="auto",
    source_mode=meta["mode"],
    extra_lines=[10, 20],
    extra_line_labels=["10 km/h max reduction (High Feasibility)", 
                       "20 km/h max reduction (Moderate Feasibility)"],
    extra_line_colors=["orange", "red"],
    show_mean=True,
)

In [ ]:
practical_limits = {
  "Average vehicle speeds capped at 50 km/h": {"x": 20,  "absolute": True},
}

practical_device_changes = kpi_run(best_xgb_model, X, 
    ["Average vehicle speeds capped at 50 km/h"], df_device=df, practical_limits=practical_limits)

In [ ]:
practical_limits = {
  "Average wait time capped at 40 seconds": {"x": 30,  "absolute": False},  # 8% of device mean
  "Average vehicle speeds capped at 50 km/h": {"x": 20, "absolute": True},  # 5 km/h max reduction
}

kpi_run(best_xgb_model, X, ["Average wait time capped at 40 seconds", "Average vehicle speeds capped at 50 km/h"], df_device=df, 
        gating="Intersect", gate_threshold=0.5)

In [ ]:
practical_limits = {
  "Average wait time capped at 40 seconds": {"x": 15,  "absolute": False},
  "Average vehicle speeds capped at 50 km/h": {"x": 10, "absolute": True},
}

kpi_run(best_xgb_model, X, ["Average wait time capped at 40 seconds", "Average vehicle speeds capped at 50 km/h"], df_device=df, 
        gating="Intersect", gate_threshold=0.5)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Scenario group labels (exactly as before) ---
scenario_groups = [
    "40-Second pedestrian wait time limit",
    "50 km/h vehicle speed limit",
    "40-Second pedestrian wait time limit\nand 50 km/h vehicle speed limit"
]

# --- Practicality levels (ordered High → Moderate → Theoretical Maximum) ---
levels = [
    "High Feasibility Scenario",        # strict
    "Moderate Feasibility Scenario",    # moderate
    "Unconstrained Scenario"       # unconstrained
]

# --- Mean Estimated Volume Increase (%) from your table ---
# Order: [High Practicality, Moderate Practicality, Theoretical Maximum]
means = np.array([
    [0.1,  1.5,  5.4],    # Wait time: 15% cap, 30% cap, no limit
    [6.8, 10.1,  8.5],    # Vehicle speeds: 10 km/h cap, 20 km/h cap, no limit
    [11.6, 11.6, 11.6]    # Combined: low, mid, no limit
])

# --- Plot settings ---
x = np.arange(len(scenario_groups))
width = 0.22

# Consistent colors across all groups
level_colors = {
    "High Feasibility Scenario": "#F28E2B",       # orange
    "Moderate Feasibility Scenario": "#E15759",   # red
    "Unconstrained Scenario": "#4E79A7"      # blue
}

plt.figure(figsize=(10, 5))

# --- Grouped bar plot ---
for i, level in enumerate(levels):
    offset = (i - 1) * width   # ensures left→right ordering
    bars = plt.bar(
        x + offset,
        means[:, i],
        width,
        label=level,
        color=level_colors[level]
    )

    # Add value labels above bars
    for bar, val in zip(bars, means[:, i]):
        plt.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.3,
            f"{val:.1f}%",
            ha="center",
            va="bottom",
            fontsize=11
        )

plt.ylabel("Mean Estimated Pedestian Volume Increase (%)", fontsize=12)
plt.xticks(x, scenario_groups, fontsize=11)
plt.ylim(0, means.max() + 3)

plt.legend(fontsize=11, loc="upper left")
plt.tight_layout()
plt.show()